In [1]:
# --- 1. Setup and Imports ---
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
import warnings

# Suppress warnings
warnings.filterwarnings("ignore", category=UserWarning)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# --- 2. Mount Google Drive ---
try:
    drive.mount('/content/drive', force_remount=True)
    print("--- Drive Mount Successful ---")
except Exception as e:
    print(f"--- ERROR Mounting Drive: {e} ---")
    raise SystemExit("Drive mounting failed. Cannot proceed.")

# --- 3. Main Script to Generate Zoomed Plots ---
print("\n" + "="*50)
print("STARTING ZOOMED PLOT GENERATION")
print("This will only add new '..._ZOOMED.png' files.")
print("="*50 + "\n")

# --- Configuration ---
MODELS_BASE_DIR = "/content/drive/My Drive/LVEF Prediction Models"
# Define the specific models you want to process
TARGET_MODELS = {"Two-Stream-B1", "Two-Stream-L1"}

# --- Find Target Model Folders ---
target_folders = []
print(f"Searching for target models in: {MODELS_BASE_DIR}")
for root, dirs, files in os.walk(MODELS_BASE_DIR):
    run_name = os.path.basename(root)
    if run_name in TARGET_MODELS:
        # Check if it's a model folder (we'll use best_model.keras as the marker)
        if "best_model.keras" in files:
            target_folders.append((root, run_name))
            print(f"Found target: {run_name}")
    # Don't descend into sub-folders of a found model
    if run_name in TARGET_MODELS:
        dirs[:] = []

print(f"\nFound {len(target_folders)} target model folder(s) to process.")

# --- Process Each Target Model ---
for experiment_dir, run_name in target_folders:
    print(f"\n--- Processing: [{run_name}] ---")

    try:
        # Define file paths
        log_file_path = os.path.join(experiment_dir, "training_log.csv")
        new_plot_path = os.path.join(experiment_dir, "training_history_plot_ZOOMED.png")

        # Non-destructive check: Skip if zoomed plot already exists
        if os.path.exists(new_plot_path):
            print(f"INFO: 'training_history_plot_ZOOMED.png' already exists. Skipping.")
            continue

        # Check for source log file
        if not os.path.exists(log_file_path):
            print(f"ERROR: Cannot find 'training_log.csv' in folder. Skipping.")
            continue

        print("INFO: Loading 'training_log.csv'...")
        history_df = pd.read_csv(log_file_path)

        # Check for necessary columns
        if 'loss' not in history_df.columns or 'val_loss' not in history_df.columns:
            print(f"ERROR: Log file is missing 'loss' or 'val_loss' columns. Skipping.")
            continue

        # --- KEY MODIFICATION: Filter data to start from Epoch 2 ---
        # .iloc[2:] selects from the 3rd row (index 2) onwards.
        zoomed_df = history_df.iloc[2:].copy()

        if zoomed_df.empty:
            print("ERROR: No data remaining after filtering (Epoch 2). Model may have trained for < 2 epochs. Skipping.")
            continue

        print("INFO: Generating new zoomed plot...")

        # --- Robust Plotting Logic ---
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))

        # Plot using the index of the *zoomed* dataframe.
        # This ensures the x-axis starts at 2.
        ax.plot(zoomed_df.index, zoomed_df['loss'], label='Training Loss')
        ax.plot(zoomed_df.index, zoomed_df['val_loss'], label='Validation Loss')

        ax.set_title(f'Training History for {run_name} (Zoomed from Epoch 2)')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss (MSE)')
        ax.legend()
        ax.grid(True)

        # Save the new, separate plot
        fig.savefig(new_plot_path)
        plt.close(fig)

        print(f"SUCCESS: Generated 'training_history_plot_ZOOMED.png'.")

    except Exception as e:
        print(f"CRITICAL FAILURE processing folder '{run_name}'. Reason: {e}")

print("\n" + "="*50)
print("Zoomed Plot Generation Complete.")
print("="*50)

Mounted at /content/drive
--- Drive Mount Successful ---

STARTING ZOOMED PLOT GENERATION
This will only add new '..._ZOOMED.png' files.

Searching for target models in: /content/drive/My Drive/LVEF Prediction Models
Found target: Two-Stream-L1
Found target: Two-Stream-B1

Found 2 target model folder(s) to process.

--- Processing: [Two-Stream-L1] ---
INFO: Loading 'training_log.csv'...
INFO: Generating new zoomed plot...
SUCCESS: Generated 'training_history_plot_ZOOMED.png'.

--- Processing: [Two-Stream-B1] ---
INFO: Loading 'training_log.csv'...
INFO: Generating new zoomed plot...
SUCCESS: Generated 'training_history_plot_ZOOMED.png'.

Zoomed Plot Generation Complete.
